In [34]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mahdimaktabdar/chatgpt-classification-dataset/sentence_level_data.csv
/kaggle/input/datasets/mahdimaktabdar/chatgpt-classification-dataset/article_level_data.csv


# import 

In [35]:
import os
import gc
import math
import random
import numpy as np
import pandas as pd
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoConfig,
    get_linear_schedule_with_warmup
)


# 2) Reproducibility


In [36]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

# 3) Load datasets


In [37]:
SENTENCE_PATH = "/kaggle/input/datasets/mahdimaktabdar/chatgpt-classification-dataset/sentence_level_data.csv"
ARTICLE_PATH = "/kaggle/input/datasets/mahdimaktabdar/chatgpt-classification-dataset/article_level_data.csv"


In [38]:
sent_df = pd.read_csv(SENTENCE_PATH)
art_df = pd.read_csv(ARTICLE_PATH)

print("Sentence-level shape:", sent_df.shape)
print("Article-level shape:", art_df.shape)

print(sent_df.head())
print(art_df.head())


Sentence-level shape: (7344, 3)
Article-level shape: (1018, 3)
   Unnamed: 0                                           sentence  class
0           0  NLP is a multidisciplinary field that draws fr...      0
1           1  In terms of linguistics, a program must be abl...      0
2           2  Of course each language has its own forms of a...      0
3           3  Programs can use several strategies for dealin...      0
4           4  As formidable as the task of extracting the co...      0
   Unnamed: 0                                            article  class
0           0  NLP is a multidisciplinary field that draws fr...      0
1           1  There are a variety of emerging applications f...      0
2           2  As each new means of communication and social ...      0
3           3  These suggestions include:, Learn about the pu...      0
4           4  In recent years there has been growing concern...      0


# 4) Robust column detection


In [39]:
# This is a helper function which can be handy for future

def detect_text_label_columns(df):
    cols = [c.lower() for c in df.columns]
    original = list(df.columns)

    text_candidates = ["text", "sentence", "article", "content", "response"]
    label_candidates = ["label", "class", "target", "generated"]

    text_col = None
    label_col = None

    for cand in text_candidates:
        for i, c in enumerate(cols):
            if cand == c or cand in c:
                text_col = original[i]
                break
        if text_col is not None:
            break

    for cand in label_candidates:
        for i, c in enumerate(cols):
            if cand == c or cand in c:
                label_col = original[i]
                break
        if label_col is not None:
            break

    if text_col is None:
        for c in original:
            if df[c].dtype == object:
                text_col = c
                break

    if label_col is None:
        for c in original:
            if pd.api.types.is_numeric_dtype(df[c]):
                uniq = sorted(df[c].dropna().unique().tolist())
                if set(uniq).issubset({0, 1}):
                    label_col = c
                    break

    return text_col, label_col


# 5)Basic cleaning

In [40]:
sent_text_col, sent_label_col = detect_text_label_columns(sent_df)
art_text_col, art_label_col = detect_text_label_columns(art_df)

print("Sentence-level:", sent_text_col, sent_label_col)
print("Article-level:", art_text_col, art_label_col)


Sentence-level: sentence class
Article-level: article class


In [41]:
def prepare_dataframe(df, text_col, label_col):
    df = df[[text_col, label_col]].copy()
    df.columns = ["text", "label"]
    df["text"] = df["text"].astype(str).str.strip()
    df["label"] = df["label"].astype(int)
    df = df.dropna()
    df = df[df["text"].str.len() > 0].reset_index(drop=True)
    return df

sent_df = prepare_dataframe(sent_df, sent_text_col, sent_label_col)
art_df = prepare_dataframe(art_df, art_text_col, art_label_col)

print(sent_df["label"].value_counts())
print(art_df["label"].value_counts())


label
1    4008
0    3336
Name: count, dtype: int64
label
0    509
1    509
Name: count, dtype: int64


# 6) Train/validation/test split


In [42]:
def stratified_split(df, test_size=0.15, val_size=0.15, seed=42):
    train_df, test_df = train_test_split(
        df,
        test_size=test_size,
        stratify=df["label"],
        random_state=seed
    )
    train_df, val_df = train_test_split(
        train_df,
        test_size=val_size / (1 - test_size),
        stratify=train_df["label"],
        random_state=seed
    )
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

sent_train, sent_val, sent_test = stratified_split(sent_df)
art_train, art_val, art_test = stratified_split(art_df)

print(len(sent_train), len(sent_val), len(sent_test))
print(len(art_train), len(art_val), len(art_test))


5140 1102 1102
712 153 153


# 7) Metrics and evaluation


In [43]:
def compute_metrics(y_true, y_pred, y_prob=None):
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred)

    metrics = {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "confusion_matrix": cm
    }

    if y_prob is not None:
        try:
            metrics["roc_auc"] = roc_auc_score(y_true, y_prob)
        except:
            metrics["roc_auc"] = None
    else:
        metrics["roc_auc"] = None

    return metrics


def print_metrics(title, metrics, y_true=None, y_pred=None):
    print(f"\n===== {title} =====")
    print(f"Accuracy : {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall   : {metrics['recall']:.4f}")
    print(f"F1-score : {metrics['f1']:.4f}")
    if metrics["roc_auc"] is not None:
        print(f"ROC-AUC  : {metrics['roc_auc']:.4f}")
    print("Confusion Matrix:")
    print(metrics["confusion_matrix"])
    if y_true is not None and y_pred is not None:
        print("\nClassification Report:")
        print(classification_report(y_true, y_pred, digits=4))


# 8) Dataset classes


In [44]:
class TransformerTextDataset(Dataset):
    # may be need to change this line of code.
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


# 8.2 BiLSTM vocab + dataset


In [45]:
from collections import Counter
from torch.nn.utils.rnn import pad_sequence


In [46]:
class Vocab:
    def __init__(self, min_freq=2, specials=["<pad>", "<unk>"]):
        self.min_freq = min_freq
        self.specials = specials
        self.stoi = {}
        self.itos = []

    def build(self, texts):
        counter = Counter()
        for text in texts:
            counter.update(text.lower().split())

        self.itos = list(self.specials)
        for token, freq in counter.items():
            if freq >= self.min_freq:
                self.itos.append(token)

        self.stoi = {tok: idx for idx, tok in enumerate(self.itos)}

    def encode(self, text):
        unk = self.stoi["<unk>"]
        return [self.stoi.get(tok, unk) for tok in text.lower().split()]

    def __len__(self):
        return len(self.itos)


In [47]:
class LSTMTextDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.vocab = vocab

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids = self.vocab.encode(self.texts[idx])
        if len(ids) == 0:
            ids = [self.vocab.stoi["<unk>"]]
        return torch.tensor(ids, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.float)


#  9) Model 1: BiLSTM


In [48]:
def lstm_collate_fn(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(x) for x in sequences], dtype=torch.long)
    padded = pad_sequence(sequences, batch_first=True, padding_value=0)
    labels = torch.stack(labels)
    return padded, lengths, labels


In [49]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden1=128, hidden2=64, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)

        self.lstm1 = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden1,
            batch_first=True,
            bidirectional=True
        )
        self.lstm2 = nn.LSTM(
            input_size=hidden1 * 2,
            hidden_size=hidden2,
            batch_first=True,
            bidirectional=True
        )

        self.fc1 = nn.Linear(hidden2 * 2, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.fc_out = nn.Linear(128, 1)

    def forward(self, input_ids, lengths):
        x = self.embedding(input_ids)
        x, _ = self.lstm1(x)
        x, (h_n, _) = self.lstm2(x)

        forward_last = h_n[-2]
        backward_last = h_n[-1]
        x = torch.cat([forward_last, backward_last], dim=1)

        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        logits = self.fc_out(x).squeeze(1)
        return logits


# 10) Model 2: DistilBERT


In [50]:
class DistilBERTClassifier(nn.Module):
    def __init__(self, model_name="distilbert-base-uncased", dropout=0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.fc1 = nn.Linear(hidden_size, 512)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(512, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_rep = outputs.last_hidden_state[:, 0, :]
        x = self.fc1(cls_rep)
        x = self.relu(x)
        x = self.dropout(x)
        logits = self.out(x).squeeze(1)
        return logits


# 11) Model 3: RoBERTa


In [51]:
class RoBERTaClassifier(nn.Module):
    def __init__(self, model_name="roberta-base", dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_size, 512)
        self.relu = nn.ReLU()
        self.out = nn.Linear(512, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        rep = outputs.last_hidden_state[:, 0, :]
        x = self.dropout(rep)
        x = self.fc1(x)
        x = self.relu(x)
        logits = self.out(x).squeeze(1)
        return logits


# 12) Model 4: Custom Deep Model


In [52]:
class FactorizedEmbedding(nn.Module):
    def __init__(self, vocab_size, embedding_size, hidden_size, pad_token_id=0):
        super().__init__()
        self.word_embeddings = nn.Embedding(vocab_size, embedding_size, padding_idx=pad_token_id)
        self.embedding_projection = nn.Linear(embedding_size, hidden_size)
        self.layer_norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(0.1)

    def forward(self, input_ids):
        x = self.word_embeddings(input_ids)
        x = self.embedding_projection(x)
        x = self.layer_norm(x)
        x = self.dropout(x)
        return x


In [53]:
class PositionalEncoding(nn.Module):
    def __init__(self, hidden_size, max_len=256):
        super().__init__()
        pe = torch.zeros(max_len, hidden_size)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, hidden_size, 2).float() * (-math.log(10000.0) / hidden_size))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


In [54]:
class SharedTransformerBlock(nn.Module):
    def __init__(self, hidden_size=256, num_heads=4, intermediate_size=512, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.attn_ln = nn.LayerNorm(hidden_size)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_size, intermediate_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(intermediate_size, hidden_size),
            nn.Dropout(dropout)
        )
        self.ffn_ln = nn.LayerNorm(hidden_size)

    def forward(self, x, attention_mask=None):
        key_padding_mask = None
        if attention_mask is not None:
            key_padding_mask = attention_mask == 0

        attn_output, _ = self.attn(
            query=x,
            key=x,
            value=x,
            key_padding_mask=key_padding_mask,
            need_weights=False
        )
        x = self.attn_ln(x + attn_output)
        ffn_output = self.ffn(x)
        x = self.ffn_ln(x + ffn_output)
        return x


In [55]:
class CustomDeepModel(nn.Module):
    def __init__(
        self,
        vocab_size=30522,
        max_len=256,
        embedding_size=128,
        hidden_size=256,
        intermediate_size=512,
        num_heads=4,
        num_shared_layers=4,
        pad_token_id=0
    ):
        super().__init__()
        self.embeddings = FactorizedEmbedding(
            vocab_size=vocab_size,
            embedding_size=embedding_size,
            hidden_size=hidden_size,
            pad_token_id=pad_token_id
        )
        self.position_embeddings = PositionalEncoding(hidden_size, max_len=max_len)
        self.shared_block = SharedTransformerBlock(
            hidden_size=hidden_size,
            num_heads=num_heads,
            intermediate_size=intermediate_size,
            dropout=0.1
        )
        self.num_shared_layers = num_shared_layers

        self.dropout = nn.Dropout(0.1)
        self.fc1 = nn.Linear(hidden_size, 512)
        self.relu = nn.ReLU()
        self.out = nn.Linear(512, 1)

    def forward(self, input_ids, attention_mask=None):
        x = self.embeddings(input_ids)
        x = self.position_embeddings(x)

        for _ in range(self.num_shared_layers):
            x = self.shared_block(x, attention_mask=attention_mask)

        rep = x[:, 0, :]
        x = self.dropout(rep)
        x = self.fc1(x)
        x = self.relu(x)
        logits = self.out(x).squeeze(1)
        return logits


# 13) Training and evaluation loops


## 13.1 Generic transformer training loop


In [56]:
def train_transformer_model(
    model,
    train_loader,
    val_loader,
    epochs=4,
    lr=2e-5,
    weight_decay=1e-2,
    patience=2
):
    model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )

    best_val_f1 = -1
    best_state = None
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].float().to(device)

            optimizer.zero_grad()
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            train_losses.append(loss.item())

        val_metrics, _, _ = evaluate_transformer_model(model, val_loader)
        avg_train_loss = np.mean(train_losses)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val F1: {val_metrics['f1']:.4f}")

        if val_metrics["f1"] > best_val_f1:
            best_val_f1 = val_metrics["f1"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model


In [57]:
@torch.no_grad()
def evaluate_transformer_model(model, data_loader):
    model.eval()
    y_true, y_pred, y_prob = [], [], []

    for batch in data_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].cpu().numpy()

        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(int)

        y_true.extend(labels.tolist())
        y_prob.extend(probs.tolist())
        y_pred.extend(preds.tolist())

    metrics = compute_metrics(y_true, y_pred, y_prob)
    return metrics, y_true, y_pred


## 13.2 BiLSTM training loop


In [58]:
def train_lstm_model(
    model,
    train_loader,
    val_loader,
    epochs=8,
    lr=1e-3,
    patience=2
):
    model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_f1 = -1
    best_state = None
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for input_ids, lengths, labels in train_loader:
            input_ids = input_ids.to(device)
            lengths = lengths.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            logits = model(input_ids, lengths)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_losses.append(loss.item())

        val_metrics, _, _ = evaluate_lstm_model(model, val_loader)
        avg_train_loss = np.mean(train_losses)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val F1: {val_metrics['f1']:.4f}")

        if val_metrics["f1"] > best_val_f1:
            best_val_f1 = val_metrics["f1"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model


In [59]:
@torch.no_grad()
def evaluate_lstm_model(model, data_loader):
    model.eval()
    y_true, y_pred, y_prob = [], [], []

    for input_ids, lengths, labels in data_loader:
        input_ids = input_ids.to(device)
        lengths = lengths.to(device)

        logits = model(input_ids, lengths)
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(int)

        y_true.extend(labels.numpy().astype(int).tolist())
        y_prob.extend(probs.tolist())
        y_pred.extend(preds.tolist())

    metrics = compute_metrics(y_true, y_pred, y_prob)
    return metrics, y_true, y_pred


# 14) Runner for one dataset


In [60]:
def free_memory():
    gc.collect()
    torch.cuda.empty_cache()


In [61]:
def run_all_deep_models_on_dataset(
    train_df,
    val_df,
    test_df,
    dataset_name="sentence_level",
    max_length=256,
    batch_size_transformer=16,
    batch_size_lstm=32
):
    results = []

    print(f"\n\n========== Running on {dataset_name} ==========")

    # 1) BiLSTM
    vocab = Vocab(min_freq=2)
    vocab.build(train_df["text"])

    train_lstm_ds = LSTMTextDataset(train_df["text"], train_df["label"], vocab)
    val_lstm_ds = LSTMTextDataset(val_df["text"], val_df["label"], vocab)
    test_lstm_ds = LSTMTextDataset(test_df["text"], test_df["label"], vocab)

    train_lstm_loader = DataLoader(train_lstm_ds, batch_size=batch_size_lstm, shuffle=True, collate_fn=lstm_collate_fn)
    val_lstm_loader = DataLoader(val_lstm_ds, batch_size=batch_size_lstm, shuffle=False, collate_fn=lstm_collate_fn)
    test_lstm_loader = DataLoader(test_lstm_ds, batch_size=batch_size_lstm, shuffle=False, collate_fn=lstm_collate_fn)

    bilstm_model = BiLSTMClassifier(vocab_size=len(vocab), embed_dim=128, hidden1=128, hidden2=64)
    bilstm_model = train_lstm_model(bilstm_model, train_lstm_loader, val_lstm_loader, epochs=8, lr=1e-3, patience=2)
    bilstm_metrics, y_true, y_pred = evaluate_lstm_model(bilstm_model, test_lstm_loader)
    print_metrics(f"{dataset_name} - BiLSTM", bilstm_metrics, y_true, y_pred)
    results.append({"dataset": dataset_name, "model": "BiLSTM", **{k: v for k, v in bilstm_metrics.items() if k != "confusion_matrix"}})
    free_memory()

    # 2) DistilBERT
    distil_name = "distilbert-base-uncased"
    distil_tokenizer = AutoTokenizer.from_pretrained(distil_name)

    train_tf_ds = TransformerTextDataset(train_df["text"], train_df["label"], distil_tokenizer, max_length=max_length)
    val_tf_ds = TransformerTextDataset(val_df["text"], val_df["label"], distil_tokenizer, max_length=max_length)
    test_tf_ds = TransformerTextDataset(test_df["text"], test_df["label"], distil_tokenizer, max_length=max_length)

    train_tf_loader = DataLoader(train_tf_ds, batch_size=batch_size_transformer, shuffle=True)
    val_tf_loader = DataLoader(val_tf_ds, batch_size=batch_size_transformer, shuffle=False)
    test_tf_loader = DataLoader(test_tf_ds, batch_size=batch_size_transformer, shuffle=False)

    distil_model = DistilBERTClassifier(model_name=distil_name, dropout=0.2)
    distil_model = train_transformer_model(distil_model, train_tf_loader, val_tf_loader, epochs=4, lr=2e-5, patience=2)
    distil_metrics, y_true, y_pred = evaluate_transformer_model(distil_model, test_tf_loader)
    print_metrics(f"{dataset_name} - DistilBERT", distil_metrics, y_true, y_pred)
    results.append({"dataset": dataset_name, "model": "DistilBERT", **{k: v for k, v in distil_metrics.items() if k != "confusion_matrix"}})
    free_memory()

    # 3) RoBERTa
    roberta_name = "roberta-base"
    roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_name)

    train_tf_ds = TransformerTextDataset(train_df["text"], train_df["label"], roberta_tokenizer, max_length=max_length)
    val_tf_ds = TransformerTextDataset(val_df["text"], val_df["label"], roberta_tokenizer, max_length=max_length)
    test_tf_ds = TransformerTextDataset(test_df["text"], test_df["label"], roberta_tokenizer, max_length=max_length)

    train_tf_loader = DataLoader(train_tf_ds, batch_size=batch_size_transformer, shuffle=True)
    val_tf_loader = DataLoader(val_tf_ds, batch_size=batch_size_transformer, shuffle=False)
    test_tf_loader = DataLoader(test_tf_ds, batch_size=batch_size_transformer, shuffle=False)

    roberta_model = RoBERTaClassifier(model_name=roberta_name, dropout=0.1)
    roberta_model = train_transformer_model(roberta_model, train_tf_loader, val_tf_loader, epochs=4, lr=2e-5, patience=2)
    roberta_metrics, y_true, y_pred = evaluate_transformer_model(roberta_model, test_tf_loader)
    print_metrics(f"{dataset_name} - RoBERTa", roberta_metrics, y_true, y_pred)
    results.append({"dataset": dataset_name, "model": "RoBERTa", **{k: v for k, v in roberta_metrics.items() if k != "confusion_matrix"}})
    free_memory()

    # 4) Custom Deep Model
    bert_tokenizer_name = "bert-base-uncased"
    bert_tokenizer = AutoTokenizer.from_pretrained(bert_tokenizer_name)

    train_tf_ds = TransformerTextDataset(train_df["text"], train_df["label"], bert_tokenizer, max_length=max_length)
    val_tf_ds = TransformerTextDataset(val_df["text"], val_df["label"], bert_tokenizer, max_length=max_length)
    test_tf_ds = TransformerTextDataset(test_df["text"], test_df["label"], bert_tokenizer, max_length=max_length)

    train_tf_loader = DataLoader(train_tf_ds, batch_size=batch_size_transformer, shuffle=True)
    val_tf_loader = DataLoader(val_tf_ds, batch_size=batch_size_transformer, shuffle=False)
    test_tf_loader = DataLoader(test_tf_ds, batch_size=batch_size_transformer, shuffle=False)

    custom_model = CustomDeepModel(
        vocab_size=bert_tokenizer.vocab_size,
        max_len=max_length,
        embedding_size=128,
        hidden_size=256,
        intermediate_size=512,
        num_heads=4,
        num_shared_layers=4,
        pad_token_id=bert_tokenizer.pad_token_id
    )
    custom_model = train_transformer_model(custom_model, train_tf_loader, val_tf_loader, epochs=6, lr=2e-4, patience=2)
    custom_metrics, y_true, y_pred = evaluate_transformer_model(custom_model, test_tf_loader)
    print_metrics(f"{dataset_name} - CustomDeepModel", custom_metrics, y_true, y_pred)
    results.append({"dataset": dataset_name, "model": "CustomDeepModel", **{k: v for k, v in custom_metrics.items() if k != "confusion_matrix"}})
    free_memory()

    return pd.DataFrame(results)


# 15) Run on sentence-level and article-level


In [ ]:
sentence_results = run_all_deep_models_on_dataset(
    sent_train, sent_val, sent_test,
    dataset_name="sentence_level",
    max_length=128,
    batch_size_transformer=16,
    batch_size_lstm=32
)

article_results = run_all_deep_models_on_dataset(
    art_train, art_val, art_test,
    dataset_name="article_level",
    max_length=256,
    batch_size_transformer=8,
    batch_size_lstm=16
)




========== Running on sentence_level ==========
Epoch 1/8 | Train Loss: 0.5688 | Val F1: 0.7863
Epoch 2/8 | Train Loss: 0.4184 | Val F1: 0.7485
Epoch 3/8 | Train Loss: 0.2960 | Val F1: 0.7782
Early stopping triggered.

===== sentence_level - BiLSTM =====
Accuracy : 0.7414
Precision: 0.7540
Recall   : 0.7804
F1-score : 0.7670
ROC-AUC  : 0.8185
Confusion Matrix:
[[348 153]
 [132 469]]

Classification Report:
              precision    recall  f1-score   support

           0     0.7250    0.6946    0.7095       501
           1     0.7540    0.7804    0.7670       601

    accuracy                         0.7414      1102
   macro avg     0.7395    0.7375    0.7382      1102
weighted avg     0.7408    0.7414    0.7408      1102



config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
